In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

#improt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier


In [ ]:
#load the files
train_df=pd.read_csv('/kaggle/input/playground-series-s5e7/train.csv')
test_df=pd.read_csv('/kaggle/input/playground-series-s5e7/test.csv')

#preview of dataset
train_df.head()

In [ ]:
# Check data types of all columns
print("Data Types:")
print(train_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(train_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(train_df.describe())
print("\n")


In [ ]:
df = train_df.copy()

num_imputer = SimpleImputer(strategy='mean')
cat_imputer = SimpleImputer(strategy='most_frequent')
cat_cols = ['Stage_fear', 'Drained_after_socializing']
target_col = 'Personality'
num_cols = ['Time_spent_Alone', 'Social_event_attendance', 'Going_outside', 
            'Friends_circle_size', 'Post_frequency']
df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

encoders = {}

# Encode input categorical columns
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Encode target column
target_encoder = LabelEncoder()
df[target_col] = target_encoder.fit_transform(df[target_col])
encoders[target_col] = target_encoder

train_df = df
train_df.head()

In [ ]:
# Check data types of all columns
print("Data Types:")
print(train_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(train_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(train_df.describe())
print("\n")


In [ ]:
X = train_df.drop(columns=['id', 'Personality'])  # Features only
y = train_df['Personality']                       # Target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
cat = CatBoostClassifier(verbose=0)
lgb = LGBMClassifier(random_state=42)

xgb.fit(X_train, y_train)
cat.fit(X_train, y_train)
lgb.fit(X_train, y_train)

xgb_preds = xgb.predict_proba(X_test)
cat_preds = cat.predict_proba(X_test)
lgb_preds = lgb.predict_proba(X_test)

ensemble_preds_proba = (xgb_preds + cat_preds + lgb_preds) / 3
ensemble_preds = np.argmax(ensemble_preds_proba, axis=1)

y_true_df = pd.DataFrame({'target': y_test, 'id': X_test.index})
y_pred_df = pd.DataFrame({'target': ensemble_preds, 'id': X_test.index})

accuracy = accuracy_score(y_test, ensemble_preds)
print("Ensemble Accuracy:", accuracy)

In [ ]:
# Check data types of all columns
print("Data Types:")
print(test_df.dtypes)
print("\n")

# Check for missing values
print("Missing Values:")
print(test_df.isnull().sum())
print("\n")


# Get basic statistics of numeric columns
print("Basic Statistics:")
print(test_df.describe())
print("\n")


In [ ]:
test_df_clean = test_df.copy()

test_df_clean[num_cols] = num_imputer.transform(test_df_clean[num_cols])
test_df_clean[cat_cols] = cat_imputer.transform(test_df_clean[cat_cols])

for col in cat_cols:
    le = encoders[col]
    test_df_clean[col] = le.transform(test_df_clean[col])
test_df_clean.head()

In [ ]:
X_test_final = test_df_clean.drop(columns=['id'])

xgb_preds = xgb.predict_proba(X_test_final)
cat_preds = cat.predict_proba(X_test_final)

ensemble_proba = (xgb_preds + cat_preds) / 2
ensemble_preds = np.argmax(ensemble_proba, axis=1)

test_df_clean['Predicted_Personality'] = encoders[target_col].inverse_transform(ensemble_preds)
print(test_df_clean[['id', 'Predicted_Personality']])